<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-02-model-architecture/lesson-2.5-pre-training-at-scale/notebooks/exercises-2.5.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 2.5: Pre-Training at Scale — Practice Exercises

**Netsetos GenAI Course v2.0** | Module 2 | 8 exercises covering data quality, perplexity, TinyLM training, data mixing, CPT decisions, multi-GPU, FineWeb pipeline, and training instability simulation.

---

In [ ]:
# Install dependencies
!pip install transformers datasets tiktoken -q

---
## Exercise 1 (Easy): Data Quality Audit
Simulate 1000 web pages. Apply quality filtering (length, spam, links, repetition) + dedup. What % survives?

In [ ]:
import re, hashlib, random
random.seed(42)

def generate_page():
    templates = [
        'This is a real article about ' + random.choice(['AI','cricket','Bollywood']) + '. ' * random.randint(5,30),
        'Buy now! Click here! ' * random.randint(10,50),
        'OK',
        'http://example.com ' * random.randint(10,20),
        'the the the the ' * random.randint(20,40),
        f"{'India is a diverse country. ' * random.randint(3,15)}",
    ]
    return random.choice(templates)

pages = [generate_page() for _ in range(1000)]

def quality_filter(text):
    if len(text) < 50: return False, 'too_short'
    words = text.split()
    if len(words) < 10: return False, 'too_few_words'
    if text.count('http') > 5: return False, 'link_farm'
    if len(set(words)) / len(words) < 0.2: return False, 'repetitive'
    if 'buy now' in text.lower(): return False, 'spam'
    return True, 'pass'

results = {}
passed = []
for page in pages:
    ok, reason = quality_filter(page)
    results[reason] = results.get(reason, 0) + 1
    if ok: passed.append(page)

# Dedup
seen = set()
unique = [p for p in passed if not (hashlib.md5(p.encode()).hexdigest() in seen or seen.add(hashlib.md5(p.encode()).hexdigest()))]

print('DATA QUALITY AUDIT — 1000 Simulated Pages')
for r, c in sorted(results.items(), key=lambda x: -x[1]):
    print(f'  {r:<16} {c:>4} ({c/10:.0f}%)')
print(f'\nSurvived: {len(unique)}/1000 ({len(unique)/10:.0f}%)')
print(f'Real-world (Common Crawl → FineWeb): ~30-50%')


---
## Exercise 2 (Easy): Perplexity Calculator
Compute GPT-2 perplexity on Wikipedia, Reddit, and Python code. Which domain has lowest perplexity?

In [ ]:
import torch, math
from transformers import GPT2LMHeadModel, GPT2Tokenizer

model = GPT2LMHeadModel.from_pretrained('gpt2')
model.eval()
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

texts = {
    'Wikipedia': 'The Taj Mahal is an ivory-white marble mausoleum on the right bank of the river Yamuna in Agra, India.',
    'Reddit': 'lol this is so random haha anyone else think AI is gonna take over xD honestly idk man',
    'Python': 'def fibonacci(n):\n    if n <= 1: return n\n    return fibonacci(n-1) + fibonacci(n-2)',
}

print('PERPLEXITY COMPARISON')
for name, text in texts.items():
    ids = tokenizer(text, return_tensors='pt').input_ids
    with torch.no_grad():
        loss = model(ids, labels=ids).loss
    ppl = math.exp(loss.item())
    conf = 'HIGH' if ppl < 40 else 'MEDIUM' if ppl < 100 else 'LOW'
    print(f'  {name:<12} ppl={ppl:>8.1f}  confidence={conf}')

print('\nLower perplexity = model more confident = domain is closer to training data')


---
## Exercise 3 (Medium): Train TinyLM with Monitoring
Train a character-level GPT on Shakespeare. Log loss, gradient norm, perplexity. Generate samples at intervals.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, math

class TinyGPT(nn.Module):
    def __init__(self, vocab_size, d=64, heads=4, layers=2, maxlen=64):
        super().__init__()
        self.te = nn.Embedding(vocab_size, d)
        self.pe = nn.Embedding(maxlen, d)
        layer = nn.TransformerEncoderLayer(d, heads, d*4, batch_first=True)
        self.tf = nn.TransformerEncoder(layer, layers)
        self.head = nn.Linear(d, vocab_size)
        self.maxlen = maxlen
    def forward(self, x):
        B,T = x.shape
        x = self.te(x) + self.pe(torch.arange(T, device=x.device))
        mask = torch.triu(torch.ones(T,T,device=x.device),1).bool()
        return self.head(self.tf(x, mask=mask))

text = 'to be or not to be that is the question whether tis nobler in the mind to suffer ' * 100
chars = sorted(set(text))
stoi = {c:i for i,c in enumerate(chars)}
itos = {i:c for c,i in stoi.items()}
data = torch.tensor([stoi[c] for c in text])
SL = 48

model = TinyGPT(len(chars))
opt = torch.optim.AdamW(model.parameters(), lr=3e-4)

for epoch in range(15):
    model.train()
    tl, tg, n = 0, 0, 0
    for i in range(0, len(data)-SL-1, SL*4):
        x = data[i:i+SL].unsqueeze(0)
        y = data[i+1:i+SL+1].unsqueeze(0)
        loss = F.cross_entropy(model(x).view(-1,len(chars)), y.view(-1))
        opt.zero_grad(); loss.backward()
        gn = sum(p.grad.norm()**2 for p in model.parameters() if p.grad is not None)**0.5
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        tl += loss.item(); tg += gn.item(); n += 1
    al, ag = tl/n, tg/n
    if epoch % 4 == 0:
        model.eval()
        ids = [stoi[c] for c in 'to be']
        with torch.no_grad():
            for _ in range(40):
                logits = model(torch.tensor([ids[-SL:]]))[0,-1]/0.8
                ids.append(torch.multinomial(F.softmax(logits,-1),1).item())
        print(f'Epoch {epoch+1:>2}: loss={al:.3f} ppl={math.exp(al):.0f} grad={ag:.2f} → "{"".join(itos[i] for i in ids[5:45])}"')
    else:
        print(f'Epoch {epoch+1:>2}: loss={al:.3f} ppl={math.exp(al):.0f} grad={ag:.2f}')


---
## Exercise 4 (Medium): Data Mixing Experiment
Compare web-only vs web+code training. Does code data improve non-code reasoning?

In [ ]:
# Conceptual exercise — train two TinyLMs from Exercise 3:
#   Model A: 100% web text (Wikipedia-style)
#   Model B: 80% web + 20% Python code
#
# Compare on:
#   1. Web perplexity (Model A slightly better)
#   2. Code perplexity (Model B much better)
#   3. Simple logic tasks (Model B often better!)

print('KEY FINDING from LLaMA 3 research:')
print('  Code data (~10% of mix) improves reasoning on ALL tasks')
print('  Code is structured, logical, step-by-step → transfers to general reasoning')
print('  This is why every major LLM includes code in pre-training mix')


---
## Exercise 5 (Medium): CPT Decision Tree
For 5 real scenarios, decide: Continual Pre-Training vs SFT vs from-scratch. Justify with cost estimates.

In [ ]:
scenarios = [
    ('Medical chatbot (Indian hospitals)', 'CPT → SFT', '~$5K + $10', 'Medical domain very different from web'),
    ('Hindi translator', 'CPT on IndicCorp → SFT', '~$2K + $10', 'LLaMA has limited Hindi'),
    ('Code reviewer', 'SFT only', '~$10', 'LLaMA already trained on 10% code'),
    ('Customer support (English)', 'SFT only', '~$10', 'Standard English domain'),
    ('Indian legal Q&A', 'CPT → SFT', '~$3K + $10', 'Legal terminology is specialized'),
]

print('CPT DECISION TREE')
print(f'{"Scenario":<35} {"Decision":<22} {"Cost":<15} Reasoning')
print('-' * 100)
for name, dec, cost, reason in scenarios:
    print(f'{name:<35} {dec:<22} {cost:<15} {reason}')

print('\nRULES:')
print('  1. NEVER CPT an instruction-tuned model (loses capabilities)')
print('  2. CPT lr = 10-30x lower than initial pre-training')
print('  3. Mix: 80% domain + 20% general data to prevent forgetting')


---
## Exercise 6 (Hard): Multi-GPU on GCP
Set up Vertex AI with 2×T4 GPUs, train with DDP, measure speedup.

In [ ]:
# This exercise requires GCP access. Here's the setup:
# 1. Create Vertex AI Custom Training Job
# 2. Configure: 2× NVIDIA T4 GPUs
# 3. Run with: torchrun --nproc_per_node=2 train.py

# DDP benchmark (simulated):
import torch, time

# Simulate single-GPU timing
model = torch.nn.Sequential(
    torch.nn.Linear(256, 1024), torch.nn.ReLU(),
    torch.nn.Linear(1024, 256)
)
opt = torch.optim.AdamW(model.parameters())
data = torch.randn(32, 256)

start = time.time()
for _ in range(200):
    loss = model(data).mean()
    opt.zero_grad(); loss.backward(); opt.step()
single = time.time() - start

print(f'Single device: {200/single:.1f} steps/sec ({single:.2f}s for 200 steps)')
print(f'Expected with 2× GPU DDP: ~{200/single*1.7:.1f} steps/sec (1.7x speedup)')
print(f'Not 2x due to gradient synchronization overhead')


---
## Exercise 7 (Hard): FineWeb Data Pipeline
Download FineWeb shard, inspect filtering, tokenize, estimate GPU-hours for 124M model.

In [ ]:
from datasets import load_dataset
import tiktoken

# Download FineWeb sample (streaming to avoid full download)
print('Loading FineWeb sample...')
ds = load_dataset('HuggingFaceFW/fineweb', name='sample-10BT', split='train', streaming=True)
samples = list(ds.take(500))

# Inspect
doc = samples[0]
print(f'Sample doc: {doc["text"][:200]}...')
print(f'URL: {doc.get("url", "N/A")}')

# Tokenize
enc = tiktoken.encoding_for_model('gpt-4o')
total_tokens = sum(len(enc.encode(s['text'])) for s in samples)
avg_len = total_tokens / len(samples)
print(f'\n500 docs = {total_tokens:,} tokens (avg {avg_len:.0f}/doc)')

# Cost estimate for 124M model
params = 124e6
flops = 6 * params * total_tokens
gpu_hours = flops / (312e12 * 0.45 * 3600)  # A100, 45% MFU
cost = gpu_hours * 2.21
print(f'\nTraining 124M on this shard: {gpu_hours:.4f} GPU-hrs = ${cost:.4f}')
print(f'Full FineWeb (15T tokens): ${6*params*15e12/(312e12*0.45*3600)*2.21:,.0f}')


---
## Exercise 8 (Challenge): Simulate Training Instability
Train TinyLM, inject garbage data at epoch 7, detect spike via gradient norm, recover from checkpoint.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, math, copy

# Reuse TinyGPT from Exercise 3
text = 'to be or not to be that is the question ' * 200
chars = sorted(set(text))
stoi = {c:i for i,c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text])
SL, VS = 48, len(chars)

class TinyGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.te=nn.Embedding(VS,64); self.pe=nn.Embedding(SL,64)
        layer=nn.TransformerEncoderLayer(64,4,256,batch_first=True)
        self.tf=nn.TransformerEncoder(layer,2); self.head=nn.Linear(64,VS)
    def forward(self,x):
        B,T=x.shape; x=self.te(x)+self.pe(torch.arange(T,device=x.device))
        return self.head(self.tf(x,mask=torch.triu(torch.ones(T,T,device=x.device),1).bool()))

model = TinyGPT()
opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
THRESH = 10.0
ckpt = None; ckpt_epoch = 0

print('INSTABILITY SIMULATION')
for epoch in range(12):
    model.train(); tl,tg,n = 0,0,0
    for i in range(0,len(data)-SL-1,SL*4):
        x = data[i:i+SL].unsqueeze(0)
        y = data[i+1:i+SL+1].unsqueeze(0)
        if epoch == 7:  # INJECT GARBAGE
            x = torch.randint(0,VS,x.shape)
            y = torch.randint(0,VS,y.shape)
        loss = F.cross_entropy(model(x).view(-1,VS), y.view(-1))
        opt.zero_grad(); loss.backward()
        gn = sum(p.grad.norm()**2 for p in model.parameters() if p.grad is not None)**0.5
        if gn.item() > THRESH and ckpt:
            print(f'  ⚠️  SPIKE at epoch {epoch+1}! grad={gn:.1f} → Rolling back to epoch {ckpt_epoch}')
            model.load_state_dict(ckpt['m']); opt.load_state_dict(ckpt['o'])
            for pg in opt.param_groups: pg['lr'] *= 0.5
            print(f'      LR reduced to {opt.param_groups[0]["lr"]:.6f}')
            break
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
        tl+=loss.item(); tg+=gn.item(); n+=1
    if n>0:
        al=tl/n; ag=tg/n
        s = '✅' if ag < THRESH else '⚠️'
        print(f'  {s} Epoch {epoch+1:>2}: loss={al:.3f} ppl={math.exp(min(al,20)):.0f} grad={ag:.2f}')
        if epoch%3==0 and ag<THRESH:
            ckpt = {'m':copy.deepcopy(model.state_dict()),'o':copy.deepcopy(opt.state_dict())}
            ckpt_epoch = epoch+1
            print(f'      💾 Checkpoint saved')

print('\nRecovery successful! Key lessons:')
print('  - Monitor gradient norm continuously')
print('  - Checkpoint regularly')
print('  - On spike: rollback + reduce lr')


---
**All 8 exercises complete.** 12 pre-training concepts mastered: data curation, training loop, distributed training, Vertex AI, scaling laws, training pipeline, open datasets, cost estimation, data mixing, instabilities, CPT, and evaluation.